# Boosting

Boosting
: Combines many weak classifiers (that are slightly better than random guessing) to produce one powerful committee. 
- Weak classifiers included in the final model do not have equal weights. 

$$G(x) = \operatorname{sign}\!\left(\sum_{m=1}^{M} a_m \, G_m(x)\right),$$
$$G_m(x) = \text{ a weak learner, } a_m = \text{ weights}$$

## AdaBoost
- Adaboost is one of the first boosting model with binary targets, $(-1,1)$.
- Fit a weak classifier, $G_m(x)$ on weighted data, then increase weights on misclassified points so that the next round focuses on them. 
### Process
1. Initailize weights, $w_i=\frac{1}{N}, i=1, ...N$
2. For $m_1$ to $M$:
  <br>a. Fit a classifier $G_m(x)$ to the training data using weights $w_i$. 
  <br>b. Compute $$\mathrm{err}_m \;=\; \frac{\sum_{i=1}^{N} w_i \, I\!\big(y_i \neq G_m(x_i)\big)}{\sum_{i=1}^{N} w_i}$$
     - This is the _weighted fraction_ of incorrectly classified observations.

    c. Compute $$\alpha_m \;=\; \log\!\left(\frac{1-\mathrm{err}_m}{\mathrm{err}_m}\right)$$
    d. Set $$w_i \leftarrow w_i \exp\!\left[\alpha_m \, I\!\big(y_i \neq G_m(x_i)\big)\right]$$
3. Final classification: $$G(x) \;=\; \operatorname{sign}\!\left(\sum_{m=1}^{M} \alpha_m \, G_m(x)\right)$$

- If classification is done correctly, the weight of an observation does not change. 
- If classification is done incorrectly, the weight is increased by multiplying with $\exp{(\alpha_m)}$.
  - $\alpha$ varies with the degree of misclassification. 
  - $\alpha$ is _large_ if the error is _small_.

:::{dropdown} Adaboost Example
|  x  |  y  | m=0 w  | m=1 w  |
|-----|-----|--------|--------|
|  0  | -1  | 0.1667 | =w     |
|  3  | +1  | 0.1667 | =w     |
|  5  | -1  | 0.1667 | =w*2   |
|  8  | -1  | 0.1667 | =w*2   |
|  8  | +1  | 0.1667 | =w     |
| 10  | +1  | 0.1667 | =w     |

For $m=1$, assume that we have a tree stump.
$$
G_1(x)=
\begin{cases}
-1 & \text{if } x < 1.5 \\
+1 & \text{if } x \ge 1.5
\end{cases}
$$

$$\mathrm{err}_m \;=\; \frac{\sum_{i=1}^{N} w_i \, I\!\big(y_i \neq G_m(x_i)\big)}{\sum_{i=1}^{N} w_i} \rightarrow \text{err}_1 = \frac{1}{6} \times 2 = \frac{1}{3}$$

| x  | y  | G₁(x) | correct? |
|----|----|--------|----------|
| 0  | -1 | -1     | Yes       |
| 3  | +1 | +1     | Yes      |
| 5  | -1 | +1     | No      |
| 8  | -1 | +1     | No       |
| 8  | +1 | +1     | Yes       |
| 10 | +1 | +1     | Yes      |

$x=5$ and $x=8$ are misclassified. Retrieve weights (from $m=0$) of misclassified points. 

$$\text{err}_1 = \frac{\frac{1}{6} + \frac{1}{6}}{\frac{1}{6} \times 6} = \frac{1}{3}$$
$$\alpha_1 = \log\left(\frac{1 - \text{err}_m}{\text{err}_m}\right) = \log\left(\frac{1 - \frac{1}{3}}{\frac{1}{3}}\right) = \log(2)$$

Now, increase the weights of $x=5$ and $x=8$ by $\exp(\alpha_1) = \exp\left(\log(2)\right) = 2$.
Incorrectly classified items now have higher weights, so the next classifier is forced to pay more attention to them.

:::

## Forward Stagewise Additive Modeling
$$f(x) = \sum^M_{m=1}\beta_mb(x;\gamma_m),$$
$$b(\space) = \text{ base functions of weak learners, } \gamma = \text{ a parameter that tunes the basis function.}$$
- Generally, boosting fits an additive model. Unlike Adaboost, it is not limited to 2-class classification.
- The formula above is a re-written Adaboost classifier.


### Estimation
- A _loss function_ has to be minimized:
$$\min_{\{\beta_m, \gamma_m\}_1^M} \sum_{i=1}^{N} L\left(y_i, \sum_{m=1}^{M} \beta_m b(x; \gamma_m)\right),$$
$$L = \text{ loss function}$$
- For example, if we set squared error loss as the loss function, 
$$\min_{\{\beta_m, \gamma_m\}_1^M} \sum_{i=1}^{N} \left(y_i - \sum_{m=1}^{M} \beta_m b(x; \gamma_m)\right)^2$$
- In each iteration, we find the best fit to the _residual_ from the previous iteration.

Loss Function
: How wrong are we to predict $f_{m-1}(x_i) + \beta_m b(x_i;\gamma_m)$ when the truth is $y_i$? 

#### MART Process
1. Initialize $f_0(x) = 0$
2. For $m=1$ to $M$ (total number of iterations),
  <br>Add to the existing model such that the loss function is minimized.

   a. Minimize the loss $\sum^{N}_{i=1}L[y_i, f_{m-1}(x_i) + \beta_mb(x_i;\gamma_m)]$.
   <br>b. Update the function $f_m = f_{m-1}(x) + \beta_mb(x;\gamma_m)$.
      - $b(x;\gamma_m)$: The _new_ weak learner fitted this round.
      - $\beta_m$: Learning rate or step size for $m$. 
      - $\gamma_m$: The parameters of the new weak learner. 

## Logit Boost
Logit Boost
: A boosting algorithm that uses deviance (log likelihood) as the loss function instead of the exponential loss used in AdaBoost. Outputs a probability between $0, 1$ instead of class labels as in AdaBoost.
- **Forward stagewise** algoirthm requires an estimate $f_{m-1}(x_i)$.
- Loss function used is the deviance (negative binomial likelihood).
$$\log L = \sum_{i=1}^{N} \left[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \right]$$
- First, derive $p$ as an expression of $f(x)$. 

$$
\begin{aligned}
\log\left(\frac{p}{1-p}\right) &= f(x) \\
\frac{p}{1-p} &= e^{f(x)} \\
p &= (1-p)e^{f(x)} \\
p &= e^{f(x)} - p e^{f(x)} \\
p + p e^{f(x)} &= e^{f(x)} \\
p(1 + e^{f(x)}) &= e^{f(x)} \\
p &= \frac{e^{f(x)}}{1 + e^{f(x)}}
\end{aligned}
$$

- Rewrite the log likelihood:

$$
\begin{aligned}
p &= \frac{e^{f(x)}}{1 + e^{f(x)}} \\
\ell(y) &\propto \sum_{i=1}^{n} \left[y_i \log(p_i) + (1-y_i)\log(1-p_i)\right] \\
&= \sum_{i=1}^{n} \Bigg[
y_i f(x_i) - y_i \log\!\big(1+e^{f(x_i)}\big)
+ (1-y_i)\log\!\left(\frac{1}{1+e^{f(x_i)}}\right)
\Bigg] \\
&= \sum_{i=1}^{n} \Big[
y_i f(x_i) - \log\!\big(1+e^{f(x_i)}\big)
\Big]
\end{aligned}
$$

- $y$ is coded as $0,1$. 

- Ultimately, 
$$f_m \sim f_{m-1}(x_i)+\beta_mb(x_i;\gamma_m)$$

## MART
Multiple Additive Regression Trees
: Gradient boosting where weak learners are decision trees specifically. 

$$\beta_mb(x_i;\gamma_m) = f_m(x_i) - f_{m-1}(x_i)$$
Assuming that trees ($b(x_i;\gamma_m)$) are being used as weak learners,
1. Initialize $f_0$
2. Fir $m_1$ to $M$:
   <br>a. Compute pseudo residuals.
   <br>b. Fit a regression tree (with $J$ terminal nodes) to pseudo residuals.
   <br>c. For each terminal node, compute fitted values.
   <br>d. Update $f_m=f_{m-1} + \text{ fitted values}$.
3. Output $f_m$ as estimate of $f$. 

- Loss function is minimized by going in the opposite direction of the gradient. 
$$r_{im} = -\left[-\frac{\partial l(y_i, f(x_i))}{\partial f(x_i)}\right]_{f = f_{m-1}}$$

:::{tip} MART with Logit Boost Example
:class: dropdown

| x0 | x1 | y |
|----|----|---|
| 1  | 0  | 1 |
| 1  | 2  | 1 |
| 1  | 2  | 1 |
| 1  | 1  | 0 |
| 1  | 1  | 0 |

**$m=0$: Initialize $f_0$**
For logit boost, $f_0 = \log(\frac{\pi}{1-pi})$ where the proportion is estimated as $\pi = \bar{y}$.
$$p = \frac{3}{5} = 0.6$$
$$f_0 = \log\left(\frac{\pi}{1-\pi}\right) = \log\left(\frac{0.6}{1-0.6}\right) \approx 0.405$$

**$m=1$: Calculate psuedo-residuals $\gamma$**
$$r_{im} = -\left[-\frac{\partial l(y_i, f(x_i))}{\partial f(x_i)}\right]_{f=f_{m-1}(x_i)}$$

$$= \frac{\partial\left(\sum_{j=1}^{N} \left[y_j f(x_j) - \log(1 + e^{f(x_j)})\right]\right)}{\partial f(x_i)}$$

$$= \left(y_i - \frac{e^{f(x_i)}}{1+e^{f(x_i)}}\right) = y_i - p_i$$

Since $f_0 = 0.405$, convert it back to probability as $\frac{e^{0.405}}{1+e^{0.405}} = 0.6$. Every observation has the same $p=0.6$ at this point.
- $y=1$ observations: $\text{resid} = 1-0.6=0.4$
- $y=0$ observations: $\text{resid} = 0-0.6=-0.6$

| x0 | x1 | y | p   | resid |
|----|----|---|-----|-------|
| 1  | 0  | 1 | 0.6 | 0.4   |
| 1  | 2  | 1 | 0.6 | 0.4   |
| 1  | 2  | 1 | 0.6 | 0.4   |
| 1  | 1  | 0 | 0.6 | -0.6  |
| 1  | 1  | 0 | 0.6 | -0.6  |

**$m=1$: Fit a residual tree to the residuals**
- Need to determine the variable split that reduces the error most. 
- Split on a **least squares** tree. 

$$
\sum_{k \in \text{mother}} \left(r_k - \bar{r}_{\text{mother}}\right)^2
-
\left(
\sum_{k \in \text{child1}} \left(r_k - \bar{r}_{\text{child1}}\right)^2
+
\sum_{k \in \text{child2}} \left(r_k - \bar{r}_{\text{child2}}\right)^2
\right)
$$

- This is the error after the split in the two child nodes. 

**$m=1$: Assume $x_1 < 1.5$ is the best split**

| x0 | x1 | y | p   | resid | Leaf |
|----|----|---|-----|-------|------|
| 1  | 0  | 1 | 0.6 | 0.4   | 1    |
| 1  | 2  | 1 | 0.6 | 0.4   | 2    |
| 1  | 2  | 1 | 0.6 | 0.4   | 2    |
| 1  | 1  | 0 | 0.6 | -0.6  | 1    |
| 1  | 1  | 0 | 0.6 | -0.6  | 1    |

**$m=1$: Update $f_m$**
$$\gamma = \frac{\sum_{x_i \in R_{jm}}(y_i-p(x_i))}{\sum_{x_i \in R_{jm}}p(x_i)(1-p(x_i)}$$
$$f_m(x_i) = f_{m-1}(x_i) + \gamma$$
$$\text{Leaf 1: }f_1 = 0.405 + (-1.11) = -0.7055$$
$$\text{Leaf 2: }f_1 = 0.405 + (+1.667) = +2.071$$

**$m=1$: Change back into probabilities**
$$\log(\frac{p}{1-p}) = f(x)$$
$$\text{Leaf 1: } p_1 = \frac{\exp(-0.7055)}{1+\exp(-0.7055)} \approx 0.33$$
$$\text{Leaf 2: } p_2 = \frac{\exp(2.071)}{1+\exp(2.071)} \approx 0.89$$

| x0 | x1 | y | Leaf | f1      | p1   |
|----|----|---|------|---------|------|
| 1  | 0  | 1 | 1    | -0.7055 | 0.33 |
| 1  | 2  | 1 | 2    | 2.071   | 0.89 |
| 1  | 2  | 1 | 2    | 2.071   | 0.89 |
| 1  | 1  | 0 | 1    | -0.7055 | 0.33 |
| 1  | 1  | 0 | 1    | -0.7055 | 0.33 |

:::

::::{grid} 2
:::{card}
:header: Usage 
- Data with nonlinear relationships.
- Mixed type features.
- Outliers and noise are present.
- Need to look at the feature importance.
:::
:::{card}
:header: Limitations 
- Not quite interpretable 
- Data is extremely high-dimensional and sparse. 
- When a fast prediction is wanted. 
  - Random forest prediction can be slow because it evaluates many trees. 
- Time series and ordered data without careful splitting. 
:::
::::

## XGBoost

XGBoost
: Optimized and extended implementation of gradient boosting. Uses both first and second derivatives.

### Process
1. Initialization ($f_0 = 0.5$)
4. Same as regular MART until fitting a tree with $J$ leaves. Fit a _regularized tree_ with a specifieid depth and allows for pruning. 
5. Fitted values are computed differently because of the regularization.

$$\left[
\sum_{i=1}^{n} l\!\left(y_i;\, f_{m-1}(x_i) + \gamma_{im}\right)
\right]
+ 0.5\,\nu \sum_{j=1}^{T} \gamma_{jm}^{2}
+ \alpha T$$
| Symbol | Meaning |
|--------|---------|
| $l$ | Loss function. |
| $\gamma^2$ | Squared (L2) penalty that discourages large leaf weights, similar to ridge regularization. |
| $\alpha$ | Complexity penalty that encourages pruning (fewer leaves). It does not affect split selection or the estimation of $\gamma$ for a fixed tree. |

#### Taylor Series
Taylor Series
: A way to approximate a function using a _polynomial_ built from the function's derivates at a point. 
- If $f$ is smooth enough, then $f(x)=\sum_{n=0}^{\infty}\frac{f^{(n)}(a)}{n!}(x-a)^n$.
  - **Smooth enough** means $f$ has _enough derivatives_ for the Taylor expansion. 

##### 1st Order Approximation
In order to start approximating, the function $f$ needs to be differentiable at or near $a$.
$$f(x) \approx f(a) + f'(a)(x-a)$$

##### 2nd Order Approximation
In order to continue with approximation, $f(a)$ needs to exist (twice differentiable near $a$).
$$f(x) \approx f(a) + f('a)(x-a) + \frac{f''(a)}{2}(x-a)^2$$

### Loss Approximation with Taylor Series
Second order Taylor series expansion of the first term in regards to $\sum_{i=1}^{n} l\!\left(y_i;\, f_{m-1}(x_i) + \gamma_{im}\right)$: 

$$l(y, f_{m-1}(x_i) + \gamma_{im}) \approx l(y, f_{m-1}(x_i)) + \gamma_{im} g_i + 0.5 h_i \gamma_{im}^2$$

This formula is the _full objective_ that we need to minimize. 

| Symbol | Meaning |
|--------|---------|
| $g_i$ | First derivative of the loss function _with respect to_ $f_{m-1}(x_i)$. |
| $h_i$ | Second derivatives. |

$$\underbrace{\sum_{i=1}^{n} l(y_i, f_{m-1}(x_i))}_{\text{current loss (constant)}} + \underbrace{\sum_{i=1}^{n}\gamma_{im}g_i + 0.5\sum_{i=1}^{n}h_i\gamma_{im}^2}_{\text{Taylor expansion of loss}} + \underbrace{0.5\nu\sum_{j=1}^{T}\gamma_{jm}^2}_{\text{L2 regularization on leaf weights}} + \underbrace{\alpha T}_{\text{penalty on number of leaves}}$$

$$\sum_{i=1}^{n} l(y_i, f_{m-1}(x_i)) + \sum_{j=1}^{T}\left[\sum_{x_i \in R_{jm}} \gamma_{im}g_i + 0.5\left(\sum_{x_i \in R_{jm}} h_i + \nu\right)\gamma_{im}^2\right] + \alpha T$$

$$\frac{\partial}{\partial \gamma_{im}}: \quad \sum_{x_i \in R_{jm}} g_i + \left(\sum_{x_i \in R_{jm}} h_i + \nu\right)\gamma_{im} = 0$$

$$\gamma_{im} = -\frac{\sum_{x_i \in R_{jm}} g_i}{\sum_{x_i \in R_{jm}} h_i + \nu}$$

$$\gamma_{im} = \frac{\sum_{x_i \in R_{jm}} (y_i - f_{m-1}(x_i))}{n_{jm} + \nu} = \frac{\text{sum of residuals}}{\text{number of residuals} + \nu}$$

### Gaussian Outcomes
- XGBoost is a **generalization** of standard gradient boosting. 
- Gaussian outcomes lead to a squared loss function. 
- Any proposed split splits one terminal node into two child nodes.
- To evaluate the proposed split, we concentrate on the parent and child nodes only.
$$\text{similarity}= \frac{\text{sum of residuals}^2}{\text{number of residuals} + v}$$

:::{note} Adaboost Example
Similarity measures how homogenous the residuals are. 
- High similarity → Residuals consistent → Node pure → Good
- Low similarity → Residuals inconsistent → Node impure → Bad
:::

$$-\sum_{i=1}^{n} l \;+\; \left(\mathrm{similarity}_{\text{other}} + \mathrm{similarity}_{\text{parent}}\right) \;-\; \alpha T
$$
After split:
$$-\sum_{i=1}^{n} l \;+\; \mathrm{similarity}_{\text{other}} \;+\; \mathrm{similarity}_{\text{left}} \;+\; \mathrm{similarity}_{\text{right}} \;-\; \alpha (T+1)
$$
The **Gain** (Difference):
$$\operatorname{gain} = \left(\mathrm{similarity}_{\text{left}} + \mathrm{similarity}_{\text{right}} - \mathrm{similarity}_{\text{parent}}\right) - \alpha$$

::::{grid} 2
:::{grid-item}
:class: text-center
{button}`Formula Derivation <../data/Boosting/xgboost-derivation.pdf>`
:::
:::{grid-item}
:class: text-center
{button}`Example <../data/Boosting/xgboost-example.pdf>`
:::
::::

### Pruning
- $\alpha$ does not affect finding out wich split has the highest gain.
- Build a tree until some stopping criteria has been reached (_e.g.,_ maximum depth), and then prune all leaves from the bottom that have a negative $\alpha$-adjusted gain.
  - This is equivalent to pruning all leaves with a gain less than a threshold $\alpha$; $\text{gain} < \alpha$ is pruned. 

### Engineering Improvements
#### Fewer Split Points
- To determine the best split for a given tree, we need to check every threhold for every variable. (_e.g.,_ If 1000 observations, 999 potential splits). This takes a long time. Instead, consider for each variable only quantiles. 

<img src="../data/Boosting/quantile.png" style="display:block;margin:auto"/>

#### Default Missing Value
- If there is a missing value, find the split on that $x$-variable as usual on non-missing values. 
- Evaluate gain in two options: Add missing values (with non-missing $y$) to the left or the right child node.
- Add missing values to the option leading to the greater gain.
- This is mentioned as **sparsity awareness**. 

::::{grid} 2
:::{card}
:header: Usage 
_When do we use XGBoost, not standard boosting?_

- **Large dataset**: XGBoost is parallelized and cache-optimized.
- **Built-in regularization**
- **Missing data**: XGBoost handles missing values automatically by learning the best default direction at each split. 
- **Better optimization**: Second order Taylor approximation converges faster and more accurately than the first order gradient boosting.
- **Sparse Data**: XGBoost has a sparsity-aware algorithm that efficiently handles sparse feature matrices like TF-IDF vectors. 

:::
:::{card}
:header: Limitations 
_When do we NOT use XGBoost, not standard boosting?_

- Simplicity matters. 
- Small dataset.
- Less optimization needed.
- Higher interpretability needed. 
:::
::::

::::{grid} 1
:::{grid-item}
:class: text-center
{button}`Solution <../solution/Boosting.pdf>`
::::